# Scoring d'impayé (Analyse Exploratoire)

Les graphiques sont produits par des fonctions standardisées définies dans `eda_viz.py`. Le notebook se contente de les appeler.

In [1]:
import pandas as pd
import eda_viz as viz

In [2]:
df = pd.read_csv('data/data.csv')
df['subscription_datetime'] = pd.to_datetime(df['subscription_datetime'])

In [3]:
df.sample(10)

,email_domain,phone_carrier,ported_phone_number,channel,zip_code_prefix,monthly_amount,energy_type,subscription_datetime,bank_name,top_unpaid
375,ICLOUD.COM,BOUYGUES TELECOM,1.0,WEB,62.0,8.0,ELEC,2024-09-17 13:59:00,BNP PARIBAS,1
1913,GMAIL.COM,ORANGE,1.0,PHONE,16.0,4.0,ELEC,2024-11-19 14:18:00,BNP PARIBAS,0
4229,GMAIL.COM,NaN,0.0,WEB,80.0,13.0,ELEC,2024-08-02 19:26:00,NICKEL - FINANCIÈRE DES PAIEMENTS ÉLECTRONIQUES,1
4953,NaN,NaN,1.0,PHONE,81.0,17.0,ELEC,2024-11-07 09:48:00,NaN,0
1476,YAHOO.COM,BOUYGUES TELECOM,0.0,WEB,60.0,3.0,GAZ,2025-02-24 17:40:00,CIC,0
2453,SFR.FR,SFR,1.0,PHONE,94.0,7.0,ELEC,2024-09-02 16:33:00,CRÉDIT AGRICOLE,0
3628,GMAIL.COM,FREE MOBILE,1.0,WEB,35.0,8.0,ELEC,2024-12-05 10:39:00,CRÉDIT AGRICOLE,0
764,GMAIL.COM,NaN,NaN,NaN,NaN,NaN,ELEC,NaT,NaN,1
4170,GMAIL.COM,FREE MOBILE,1.0,PHONE,86.0,6.0,GAZ,2024-09-09 14:22:00,CRÉDIT MUTUEL,0
3091,HOTMAIL.COM,NaN,0.0,WEB,45.0,12.0,GAZ,2024-11-04 19:19:00,BOURSORAMA,0


In [18]:
df = df.drop_duplicates()

In [19]:
df.info()

<class 'pandas.DataFrame'>
Index: 4994 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   email_domain           4067 non-null   str           
 1   phone_carrier          4076 non-null   str           
 2   ported_phone_number    4808 non-null   float64       
 3   channel                4817 non-null   str           
 4   zip_code_prefix        4817 non-null   float64       
 5   monthly_amount         4821 non-null   float64       
 6   energy_type            4826 non-null   str           
 7   subscription_datetime  4811 non-null   datetime64[us]
 8   bank_name              4070 non-null   str           
 9   top_unpaid             4994 non-null   int64         
dtypes: datetime64[us](1), float64(3), int64(1), str(5)
memory usage: 429.2 KB


In [20]:
# Configuration centrale : on déclare les colonnes une seule fois
TARGET = 'top_unpaid'
TARGET_LABELS = {0: 'Payé', 1: 'Impayé'}
NUM_COLS = ['monthly_amount']                       # numériques continues
GEO_COL = 'zip_code_prefix'                          # géographique (≠ numérique)
CAT_COLS = ['channel', 'energy_type', 'phone_carrier', 'bank_name', 'email_domain']
CAT_COLS_TARGET = CAT_COLS + ['ported_phone_number']

## Partie I - Distributions

### 1. Équilibre des classes (variable cible)

In [21]:
viz.plot_target_balance(df, TARGET, labels=TARGET_LABELS).show()

### 2. Valeurs manquantes

In [22]:
viz.plot_missing_values(df).show()

### 3. Distributions générales
#### 3a. Variables numériques

In [23]:
for col in NUM_COLS:
    viz.plot_numeric_distribution(df, col).show()

print(df[NUM_COLS].describe().round(2))

       monthly_amount
count         4821.00
mean             7.98
std              5.14
min              0.00
25%              4.00
50%              7.00
75%             10.00
max             81.00


In [ ]:
df[df['monthly_amount']>19]

,email_domain,phone_carrier,ported_phone_number,channel,zip_code_prefix,monthly_amount,energy_type,subscription_datetime,bank_name,top_unpaid,mois_anciennete
9,GMAIL.COM,SFR,0.0,WEB,59.0,34.0,GAZ,2025-02-05 12:07:00,CAISSE D'ÉPARGNE,0,1
13,JPAYNE.CO,ORANGE,0.0,WEB,82.0,26.0,ELEC,2024-09-22 00:24:00,NaN,0,5
70,YMAIL.COM,NaN,0.0,WEB,68.0,20.0,ELEC,2024-09-18 21:35:00,NaN,0,5
206,GMAIL.COM,BOUYGUES TELECOM,1.0,WEB,61.0,21.0,ELEC,2024-09-29 11:38:00,LA BANQUE POSTALE,0,5
276,GARAGE-MOURER.FR,UNKNOWN,0.0,PHONE,57.0,45.0,ELEC,2024-09-04 14:28:00,LCL,0,6
...,...,...,...,...,...,...,...,...,...,...,...
4657,GMAIL.COM,FREE MOBILE,1.0,PHONE,62.0,25.0,ELEC,2024-11-22 16:53:00,BNP PARIBAS,0,3
4679,GMAIL.COM,ORANGE,0.0,PHONE,92.0,45.0,GAZ,2025-02-17 11:09:00,LCL,0,0
4729,GMAIL.COM,NaN,1.0,PHONE,41.0,23.0,ELEC,2024-09-04 13:53:00,CRÉDIT AGRICOLE,1,6
4764,NaN,NaN,1.0,WEB,17.0,33.0,ELEC,2024-05-25 09:32:00,CRÉDIT INDUSTRIEL DE L'OUEST,0,9


: 

In [24]:
df = df[df['monthly_amount']!=0].reset_index(drop=True)

#### 3b. Variables catégorielles

In [25]:
for col in CAT_COLS:
    viz.plot_categorical_distribution(df, col).show()

#### 3c. Variable géographique - `zip_code_prefix`

Un code géographique n'est pas une grandeur continue : on l'analyse par son taux d'impayé, pas par un histogramme. Le nuage taux/volume révèle si les zones risquées reposent sur des effectifs solides ou sur du bruit.

In [26]:
viz.plot_rate_vs_volume(df, GEO_COL, TARGET, min_count=30).show()
viz.plot_rate_by_category(df, GEO_COL, TARGET, top_n=25, min_count=30).show()

#### 3d. Variable temporelle

In [ ]:
# nb mois dans le dataset
((df['subscription_datetime'].max() - df['subscription_datetime'].min()).days)//30

10

In [28]:
viz.plot_volume_over_time(df, 'subscription_datetime').show()
viz.plot_rate_over_time(df, 'subscription_datetime', TARGET).show()

#### 3e. Ancienneté de souscription

In [29]:
df = viz.add_tenure_bins(df, 'subscription_datetime')
viz.plot_rate_by_category(df, 'tranche_anciennete', TARGET, sort='category').show()

## Partie II - Relation avec la variable cible

### 1. Variables numériques vs taux d'impayé

In [30]:
for col in NUM_COLS:
    viz.plot_numeric_by_target(df, col, TARGET, labels=TARGET_LABELS).show()

### 2. Variables catégorielles — taux d'impayé par modalité

In [31]:
for col in CAT_COLS_TARGET:
    viz.plot_rate_by_category(df, col, TARGET, min_count=30).show()

## Partie III - Colinéarité entre variables catégorielles

La V de Cramér mesure l'association entre catégorielles (0 = indépendance, 1 = association parfaite). Utile pour repérer les redondances avant la modélisation (ex. `phone_carrier` vs `ported_phone_number`).

In [32]:
viz.plot_cramers_matrix(df, CAT_COLS_TARGET).show()

### Association de chaque variable avec la cible

Classement rapide des catégorielles les plus prometteuses pour le scoring.

In [33]:
viz.cramers_with_target(df, CAT_COLS_TARGET, TARGET)

,variable,cramers_v_cible
0,bank_name,0.539
1,phone_carrier,0.211
2,email_domain,0.196
3,ported_phone_number,0.177
4,channel,0.105
5,energy_type,0.049


## Partie IV - Information Value

Pour chaque bin $i$ d'une variable :

$$WoE_i = \ln\left(\frac{D\_mauvais_i}{D\_bons_i}\right)$$

Avec :

$$D\_mauvais_i = \frac{\text{nb impayés dans le bin } i}{\text{nb impayés total}} \qquad D\_bons_i = \frac{\text{nb non-impayés dans le bin } i}{\text{nb non-impayés total}}$$

L'IV agrège la contribution de tous les bins :

$$IV = \sum_{i} \left(D\_mauvais_i - D\_bons_i\right) \times WoE_i$$

### Interprétation de l'IV

| IV | Pouvoir prédictif |
|----|-------------------|
| $< 0.02$ | Nul — variable à écarter |
| $0.02 - 0.10$ | Faible |
| $0.10 - 0.30$ | Moyen |
| $0.30 - 0.50$ | Fort |
| $> 0.50$ | Suspect — vérifier un éventuel data leakage |

### Interprétation du WoE par bin

| WoE | Signification |
|-----|---------------|
| $> 0$ | Le bin concentre plus de mauvais payeurs que la moyenne → facteur de risque |
| $\approx 0$ | Le bin ne discrimine pas → neutre |
| $< 0$ | Le bin concentre plus de bons payeurs que la moyenne → facteur protecteur |

In [34]:
# Ranking de toutes les variables
all_cols = NUM_COLS + CAT_COLS + [GEO_COL]
viz.iv_summary(df, all_cols, TARGET)

,variable,IV,pouvoir_predictif
0,bank_name,1.4866,Suspect
1,email_domain,0.3133,Fort
2,phone_carrier,0.2226,Moyen
3,zip_code_prefix,0.1374,Moyen
4,monthly_amount,0.0625,Faible
5,channel,0.0569,Faible
6,energy_type,0.0126,Nul


In [35]:
# Détail WoE des plus intéressantes
viz.plot_woe(df, "bank_name", TARGET).show()
viz.plot_woe(df, "monthly_amount", TARGET, n_bins=10).show()

## Conclusion de l'analyse exploratoire

### Qualité et structure des données
- **~4 900 souscriptions** sur **10 mois**, avec une cible **déséquilibrée** (~30 % d'impayés au global) : le déséquilibre devra être compensé à la modélisation (`class_weight` / `is_unbalance`).
- **Valeurs manquantes hétérogènes** : `email_domain`, `phone_carrier` et `bank_name` (~18-19 %) sont les plus touchées ; `ported_phone_number`, `channel`, `zip_code_prefix`, `monthly_amount` le sont marginalement (~3-4 %). → imputation explicite (`"MISSING"` + médiane) plutôt que suppression de lignes.
- **`monthly_amount`** : présence de `0` incohérents (retirés) et d'une **queue droite d'outliers** (max à 81 pour un 75ᵉ centile à 10). → justifie une troncature haute (winsorisation au 99ᵉ centile) avant modélisation.

### Pouvoir prédictif (IV) et associations (V de Cramér / cible)
| Variable | IV | V de Cramér | Lecture |
|----------|-----|-------------|---------|
| `bank_name` | **1.49** | 0.54 | IV « suspect » → **signal très fort, à surveiller pour data leakage** |
| `email_domain` | 0.31 | 0.20 | Fort, mais haute cardinalité → à transformer |
| `phone_carrier` | 0.22 | 0.21 | Moyen |
| `zip_code_prefix` | 0.14 | — | Moyen, géographique |
| `monthly_amount` | 0.06 | — | Faible |
| `channel` | 0.06 | 0.11 | Faible |
| `energy_type` | 0.01 | 0.05 | **Quasi nul → candidate à écarter** |

### Colinéarités notables
- `phone_carrier` ↔ `ported_phone_number` : association attendue (le portage dépend de l'opérateur) → redondance à garder en tête avant la modélisation.

### Décisions de feature engineering qui en découlent (cf. `scoring.ipynb`)
- **`monthly_amount`** → troncature au 99ᵉ centile (outliers).
- **`bank_name`** → regroupement sémantique (`BANK_GROUPS`) ; IV suspect → vérifier l'absence de fuite.
- **`email_domain`** → remplacée par des features dérivées (TLD, webmail, domaine français, longueur) plutôt qu'encodée brute.
- **`zip_code_prefix`** → encodé en densité urbaine (référentiel statique, sans fuite).
- **`energy_type`** → pouvoir prédictif quasi nul, candidate au retrait.

### Limites de l'analyse
- **Biais de maturité** : `top_unpaid` est cumulatif jusqu'à la date d'extraction ; les souscriptions anciennes ont une fenêtre d'observation plus longue, ce qui gonfle mécaniquement leur taux d'impayé. À garder à l'esprit pour l'interprétation des taux et pour le **split temporel** (train sur le passé, évaluation sur le futur).
